<div style="background-color:#F3F2EE">
    <br /><br />
        <p style="text-align: center;">
            <font size="6" color='#0A1781'>
                <strong>
                    Databricks Certification Learning Knowledge Graph
                </strong>
              </font>
        </p>
        <p style="text-align: center;">
            <font size="6" color='#C58A1E'>
                <strong>
                    Ontology as Data: Nodes and Relationships
                </strong>
            </font>
        </p>
        <p style="text-align: center;">
            <font size="5" color='#C58A1E'>
                <strong>
                    Iniciar a materialização da ontologia criada no Notebook 00 em estruturas tabulares reutilizáveis.
                </strong>
            </font>
        </p>
    <br />
</div>

<div style="background-color:#F3F2EE">
    <p style="text-align: right;">
      <font size="4" color='#444444'>
            Roberto SSoares - LfLngLrnng
      </font>
    </p>
    <p style="text-align: right;"><font size="2" color='#444444'>
        <a href="https://www.linkedin.com/in/roberto-dos-santos-soares/">in/roberto-dos-santos-soares</a><br /><a href="https://roberto-ssoares.github.io/meu-portfolio/">Portifólio: roberto-ssoares</a>
    </p>
    <p style="text-align: right;">
        <font size="4" color='#444444'>
            " [+] Faturamento [-] Custo [+] Qualidade de vida "
        </font>
        <br />
        <font size="2" color='#918e8e'>"Mestre Bruno Jardim"
        </font>
    </p>        
    <p style="text-align: right;">        
        <font size="2" color='#918e8e'>           
        </font>
    </p>
</div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>📌 Objetivo</strong></font>

<font size="2" color='#66666'>

>- **Ações realizadas**
    - Definição do propósito do notebook.
    - Preparação para criação dos CSVs de nós e relacionamentos.
    - Organização da ontologia como dados.

>- **Justificativa técnica**
    - Um Knowledge Graph pode ser desenhado conceitualmente, mas precisa ser materializado em estruturas persistentes para carga, auditoria, versionamento e reuso.

>- **Resultados esperados**
    - Ao final deste notebook, teremos arquivos CSV prontos para representar os nós e relacionamentos iniciais do Knowledge Graph.

---

</font></div>

In [38]:
#!uv pip install watermark -q -U
#!uv pip install tabulate -q -U

In [39]:
from datetime import date
from pathlib import Path
import tabulate

import pandas as pd

In [2]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "RobertoSSoares-LfLngLrnng"

Author: RobertoSSoares-LfLngLrnng



In [3]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [4]:
PROJECT_NAME = "Databricks Certification Learning Knowledge Graph"
CERTIFICATION_NAME = "Databricks Certified Data Engineer Associate"

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR = DATA_DIR / "exports"
DOCS_DIR = BASE_DIR / "docs"
CYPHER_DIR = BASE_DIR / "cypher"

for directory in [DATA_DIR, RAW_DIR, PROCESSED_DIR, EXPORTS_DIR, DOCS_DIR, CYPHER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BASE_DIR

WindowsPath('D:/_DS-Projects/Data-Science/databricks-learning-kg')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 1. Convenção de modelagem</strong></font>

<font size="2" color='#66666'>

>- Neste notebook, cada entidade relevante será transformada em uma tabela de nós.
    - Cada relacionamento será transformado em uma tabela de arestas.

>- A convenção usada será:

>- **Nós**
    - *`node_id`: identificador único do nó.*
    - *`node_label`: tipo semântico do nó.*
    - *`name`: nome legível.*
    - *`category`: agrupamento analítico.*
    - *`description`: descrição executiva.*
    - *`status`: estado atual.*
    - *`priority`: prioridade.*
    
>- **Relacionamentos**
    - *`source_id`: nó de origem.*
    - *`source_label`: tipo do nó de origem.*
    - *`relationship`: tipo do relacionamento.*
    - *`target_id`: nó de destino.*
    - *`target_label`: tipo do nó de destino.*
    - *`weight`: peso opcional.*
    - *`description`: descrição semântica da relação.*

---

</font></div>

In [5]:
def save_table(df: pd.DataFrame, filename: str) -> dict:
    """
    Salva um DataFrame como CSV na pasta data/processed.
    Retorna metadados básicos para o manifesto de exportação.
    """
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8")
    
    return {
        "filename": filename,
        "path": str(path),
        "rows": len(df),
        "columns": len(df.columns),
        "export_date": date.today().isoformat(),
    }


def build_node_frame(
    df: pd.DataFrame,
    id_col: str,
    label: str,
    name_col: str,
    category_col: str | None = None,
    description_col: str | None = None,
    status_col: str | None = None,
    priority_col: str | None = None,
) -> pd.DataFrame:
    """
    Padroniza diferentes tabelas de entidades em um formato único de nós.
    """
    out = pd.DataFrame()
    out["node_id"] = df[id_col].astype(str)
    out["node_label"] = label
    out["name"] = df[name_col].astype(str)
    
    out["category"] = (
        df[category_col].astype(str)
        if category_col and category_col in df.columns
        else ""
    )
    
    out["description"] = (
        df[description_col].astype(str)
        if description_col and description_col in df.columns
        else ""
    )
    
    out["status"] = (
        df[status_col].astype(str)
        if status_col and status_col in df.columns
        else ""
    )
    
    out["priority"] = (
        df[priority_col].astype(str)
        if priority_col and priority_col in df.columns
        else ""
    )
    
    return out


def make_edges(
    pairs: list[dict],
    source_label: str,
    relationship: str,
    target_label: str,
    default_weight: float = 1.0,
) -> pd.DataFrame:
    """
    Cria uma tabela padronizada de relacionamentos.
    """
    rows = []
    
    for pair in pairs:
        rows.append(
            {
                "source_id": pair["source_id"],
                "source_label": source_label,
                "relationship": relationship,
                "target_id": pair["target_id"],
                "target_label": target_label,
                "weight": pair.get("weight", default_weight),
                "description": pair.get("description", ""),
            }
        )
    
    return pd.DataFrame(rows)

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 2. Nós principais: certificação e domínios do exame</strong></font>

<font size="2" color='#66666'>

- A certificação será o nó raiz do Knowledge Graph.

- Os domínios do exame serão conectados à certificação por meio do relacionamento `HAS_DOMAIN`.

---

</font></div>

In [43]:
certifications = pd.DataFrame(
    [
        {
            "certification_id": "CERT_DEA_001",
            "name": "Databricks Certified Data Engineer Associate",
            "vendor": "Databricks",
            "level": "Associate",
            "target_role": "Data Engineer",
            "status": "planned",
            "start_date": date.today().isoformat(),
            "target_exam_date": "",
            "validity_years": 2,
            "description": "Certificação alvo para consolidar fundamentos de Engenharia de Dados na plataforma Databricks.",
        }
    ]
)

exam_domains = pd.DataFrame(
    [
        {
            "domain_id": "D01",
            "name": "Databricks Intelligence Platform",
            "weight_pct": 10,
            "priority": "medium",
            "description": "Fundamentos da plataforma, workspace, compute e arquitetura Lakehouse.",
        },
        {
            "domain_id": "D02",
            "name": "Development and Ingestion",
            "weight_pct": 30,
            "priority": "high",
            "description": "Desenvolvimento, ingestão de dados e uso de SQL/PySpark.",
        },
        {
            "domain_id": "D03",
            "name": "Data Processing & Transformations",
            "weight_pct": 31,
            "priority": "high",
            "description": "Transformações, Delta Lake, tabelas, joins, agregações e arquitetura medalhão.",
        },
        {
            "domain_id": "D04",
            "name": "Productionizing Data Pipelines",
            "weight_pct": 18,
            "priority": "high",
            "description": "Jobs, Workflows, tarefas, agendamento, monitoramento e operação.",
        },
        {
            "domain_id": "D05",
            "name": "Data Governance & Quality",
            "weight_pct": 11,
            "priority": "medium",
            "description": "Governança, Unity Catalog, qualidade, permissões e linhagem.",
        },
    ]
)


In [44]:
display(certifications)
display(exam_domains)

,certification_id,name,vendor,level,target_role,status,start_date,target_exam_date,validity_years,description
0,CERT_DEA_001,Databricks Certified Data Engineer Associate,Databricks,Associate,Data Engineer,planned,2026-04-27,,2,Certificação alvo para consolidar fundamentos de Engenharia de Dados na plataforma Databricks.


,domain_id,name,weight_pct,priority,description
0,D01,Databricks Intelligence Platform,10,medium,"Fundamentos da plataforma, workspace, compute e arquitetura Lakehouse."
1,D02,Development and Ingestion,30,high,"Desenvolvimento, ingestão de dados e uso de SQL/PySpark."
2,D03,Data Processing & Transformations,31,high,"Transformações, Delta Lake, tabelas, joins, agregações e arquitetura medalhão."
3,D04,Productionizing Data Pipelines,18,high,"Jobs, Workflows, tarefas, agendamento, monitoramento e operação."
4,D05,Data Governance & Quality,11,medium,"Governança, Unity Catalog, qualidade, permissões e linhagem."


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 3. Tópicos iniciais do KG</strong></font>

<font size="2" color='#66666'>

>- Os tópicos representam os conceitos técnicos que precisam ser estudados, praticados e evidenciados.

>- Cada tópico será conectado a um domínio oficial do exame.

---

</font></div>

In [11]:
topics_data = [
    # D01
    ("T001", "D01", "Lakehouse Architecture", "Platform", "medium", "high", 0.25, 0.10),
    ("T002", "D01", "Databricks Workspace", "Platform", "easy", "medium", 0.10, 0.00),
    ("T003", "D01", "Compute", "Platform", "medium", "medium", 0.10, 0.00),
    ("T004", "D01", "SQL Warehouse", "Platform", "medium", "medium", 0.10, 0.00),
    ("T005", "D01", "Notebooks", "Platform", "easy", "high", 0.50, 0.25),
    ("T006", "D01", "Repos", "Platform", "medium", "medium", 0.25, 0.10),

    # D02
    ("T007", "D02", "Data Ingestion", "Ingestion", "medium", "high", 0.25, 0.10),
    ("T008", "D02", "read_files", "Ingestion", "medium", "high", 0.00, 0.00),
    ("T009", "D02", "COPY INTO", "Ingestion", "medium", "medium", 0.00, 0.00),
    ("T010", "D02", "Auto Loader", "Ingestion", "hard", "medium", 0.00, 0.00),
    ("T011", "D02", "File Formats", "Ingestion", "easy", "high", 0.50, 0.25),
    ("T012", "D02", "Schema Inference", "Ingestion", "medium", "high", 0.25, 0.10),
    ("T013", "D02", "Schema Definition", "Ingestion", "medium", "high", 0.25, 0.10),
    ("T014", "D02", "Spark SQL", "Development", "medium", "high", 0.50, 0.25),
    ("T015", "D02", "PySpark DataFrame API", "Development", "medium", "high", 0.50, 0.25),

    # D03
    ("T016", "D03", "Delta Lake", "Transformation", "medium", "high", 0.25, 0.10),
    ("T017", "D03", "Delta Table", "Transformation", "medium", "high", 0.25, 0.10),
    ("T018", "D03", "Managed Table", "Transformation", "medium", "medium", 0.10, 0.00),
    ("T019", "D03", "External Table", "Transformation", "medium", "medium", 0.10, 0.00),
    ("T020", "D03", "Medallion Architecture", "Transformation", "medium", "high", 0.50, 0.25),
    ("T021", "D03", "MERGE INTO", "Transformation", "hard", "high", 0.10, 0.00),
    ("T022", "D03", "Window Functions", "Transformation", "medium", "high", 0.40, 0.20),
    ("T023", "D03", "Joins", "Transformation", "medium", "high", 0.60, 0.30),
    ("T024", "D03", "Aggregations", "Transformation", "medium", "high", 0.60, 0.30),
    ("T025", "D03", "Data Cleaning", "Transformation", "medium", "high", 0.60, 0.30),
    ("T026", "D03", "Deduplication", "Transformation", "medium", "high", 0.40, 0.20),
    ("T027", "D03", "Schema Evolution", "Transformation", "hard", "medium", 0.10, 0.00),
    ("T028", "D03", "Time Travel", "Transformation", "medium", "medium", 0.10, 0.00),

    # D04
    ("T029", "D04", "Databricks Workflows", "Production", "medium", "high", 0.00, 0.00),
    ("T030", "D04", "Jobs", "Production", "medium", "high", 0.00, 0.00),
    ("T031", "D04", "Tasks", "Production", "medium", "high", 0.00, 0.00),
    ("T032", "D04", "Task Dependencies", "Production", "medium", "high", 0.00, 0.00),
    ("T033", "D04", "Scheduling", "Production", "medium", "medium", 0.00, 0.00),
    ("T034", "D04", "Parameters", "Production", "medium", "medium", 0.00, 0.00),
    ("T035", "D04", "Monitoring", "Production", "medium", "medium", 0.00, 0.00),
    ("T036", "D04", "Retry", "Production", "medium", "medium", 0.00, 0.00),
    ("T037", "D04", "Pipeline Idempotency", "Production", "hard", "high", 0.25, 0.10),

    # D05
    ("T038", "D05", "Unity Catalog", "Governance", "medium", "high", 0.10, 0.00),
    ("T039", "D05", "Catalog", "Governance", "easy", "medium", 0.10, 0.00),
    ("T040", "D05", "Schema", "Governance", "easy", "medium", 0.25, 0.10),
    ("T041", "D05", "Table", "Governance", "easy", "medium", 0.25, 0.10),
    ("T042", "D05", "Volume", "Governance", "medium", "medium", 0.00, 0.00),
    ("T043", "D05", "Permissions", "Governance", "medium", "high", 0.10, 0.00),
    ("T044", "D05", "Lineage", "Governance", "medium", "medium", 0.10, 0.00),
    ("T045", "D05", "Data Quality", "Governance", "medium", "high", 0.40, 0.20),
    ("T046", "D05", "Naming Conventions", "Governance", "easy", "medium", 0.60, 0.30),
]

topics = pd.DataFrame(
    topics_data,
    columns=[
        "topic_id",
        "domain_id",
        "name",
        "category",
        "difficulty",
        "priority",
        "confidence_score",
        "mastery_score",
    ],
)

topics["status"] = topics["mastery_score"].apply(
    lambda x: "not_started" if x == 0 else "initial_mapping"
)

topics["exam_relevance"] = topics["priority"]
topics["portfolio_relevance"] = topics["priority"]
topics["last_review_date"] = ""

topics.head(10)

,topic_id,domain_id,name,category,difficulty,priority,confidence_score,mastery_score,status,exam_relevance,portfolio_relevance,last_review_date
0,T001,D01,Lakehouse Architecture,Platform,medium,high,0.25,0.10,initial_mapping,high,high,
1,T002,D01,Databricks Workspace,Platform,easy,medium,0.10,0.00,not_started,medium,medium,
2,T003,D01,Compute,Platform,medium,medium,0.10,0.00,not_started,medium,medium,
3,T004,D01,SQL Warehouse,Platform,medium,medium,0.10,0.00,not_started,medium,medium,
4,T005,D01,Notebooks,Platform,easy,high,0.50,0.25,initial_mapping,high,high,
5,T006,D01,Repos,Platform,medium,medium,0.25,0.10,initial_mapping,medium,medium,
6,T007,D02,Data Ingestion,Ingestion,medium,high,0.25,0.10,initial_mapping,high,high,
7,T008,D02,read_files,Ingestion,medium,high,0.00,0.00,not_started,high,high,
8,T009,D02,COPY INTO,Ingestion,medium,medium,0.00,0.00,not_started,medium,medium,
9,T010,D02,Auto Loader,Ingestion,hard,medium,0.00,0.00,not_started,medium,medium,


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 4. Subtópicos</strong></font>

<font size="2" color='#66666'>

>- Os subtópicos detalham conceitos importantes dentro de tópicos maiores.

>- Essa camada ajuda a refinar o estudo e permite que o KG identifique lacunas mais específicas.

---

</font></div>

In [12]:
subtopics = pd.DataFrame(
    [
        {"subtopic_id": "ST001", "topic_id": "T001", "name": "Data Lake vs Data Warehouse vs Lakehouse"},
        {"subtopic_id": "ST002", "topic_id": "T001", "name": "Lakehouse as unified architecture"},
        {"subtopic_id": "ST003", "topic_id": "T016", "name": "ACID transactions"},
        {"subtopic_id": "ST004", "topic_id": "T016", "name": "Delta transaction log"},
        {"subtopic_id": "ST005", "topic_id": "T016", "name": "Versioning"},
        {"subtopic_id": "ST006", "topic_id": "T020", "name": "Bronze Layer"},
        {"subtopic_id": "ST007", "topic_id": "T020", "name": "Silver Layer"},
        {"subtopic_id": "ST008", "topic_id": "T020", "name": "Gold Layer"},
        {"subtopic_id": "ST009", "topic_id": "T021", "name": "Upsert pattern"},
        {"subtopic_id": "ST010", "topic_id": "T021", "name": "Incremental load pattern"},
        {"subtopic_id": "ST011", "topic_id": "T029", "name": "Workflow orchestration"},
        {"subtopic_id": "ST012", "topic_id": "T030", "name": "Job configuration"},
        {"subtopic_id": "ST013", "topic_id": "T031", "name": "Task execution"},
        {"subtopic_id": "ST014", "topic_id": "T032", "name": "Task dependencies"},
        {"subtopic_id": "ST015", "topic_id": "T038", "name": "Centralized governance"},
        {"subtopic_id": "ST016", "topic_id": "T038", "name": "Object hierarchy"},
        {"subtopic_id": "ST017", "topic_id": "T043", "name": "Access control"},
        {"subtopic_id": "ST018", "topic_id": "T045", "name": "Quality checks"},
        {"subtopic_id": "ST019", "topic_id": "T045", "name": "Validation rules"},
        {"subtopic_id": "ST020", "topic_id": "T046", "name": "Enterprise naming standards"},
    ]
)

subtopics["description"] = "Subtópico técnico associado à jornada de certificação Databricks."
subtopics.head()

,subtopic_id,topic_id,name,description
0,ST001,T001,Data Lake vs Data Warehouse vs Lakehouse,Subtópico técnico associado à jornada de certificação Databricks.
1,ST002,T001,Lakehouse as unified architecture,Subtópico técnico associado à jornada de certificação Databricks.
2,ST003,T016,ACID transactions,Subtópico técnico associado à jornada de certificação Databricks.
3,ST004,T016,Delta transaction log,Subtópico técnico associado à jornada de certificação Databricks.
4,ST005,T016,Versioning,Subtópico técnico associado à jornada de certificação Databricks.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 5. Recursos, notebooks, labs e skills</strong></font>

<font size="2" color='#66666'>

>- Agora criamos os nós que representarão artefatos da jornada:
    - recursos de estudo;
    - notebooks;
    - labs práticos;
    - skills demonstráveis;
    - sessões de estudo;
    - snapshots de evolução.

---

</font></div>

In [13]:
resources = pd.DataFrame(
    [
        {
            "resource_id": "R001",
            "name": "Databricks Academy",
            "type": "course_platform",
            "origin": "official",
            "status": "planned",
            "description": "Plataforma oficial de cursos e trilhas de aprendizagem Databricks.",
        },
        {
            "resource_id": "R002",
            "name": "Databricks Exam Guide",
            "type": "exam_guide",
            "origin": "official",
            "status": "planned",
            "description": "Guia oficial de tópicos e estrutura da certificação.",
        },
        {
            "resource_id": "R003",
            "name": "Databricks Documentation",
            "type": "documentation",
            "origin": "official",
            "status": "planned",
            "description": "Documentação técnica da plataforma Databricks.",
        },
        {
            "resource_id": "R004",
            "name": "Notebook 00 — Domain Understanding",
            "type": "internal_notebook",
            "origin": "project",
            "status": "completed",
            "description": "Notebook conceitual inicial do projeto Databricks Learning KG.",
        },
    ]
)

notebooks = pd.DataFrame(
    [
        {
            "notebook_id": "NB00",
            "name": "00_domain_understanding_databricks_learning_kg.ipynb",
            "stage": "Domain Understanding",
            "status": "completed",
            "description": "Define a ontologia inicial e o enquadramento CRISP-DM+.",
        },
        {
            "notebook_id": "NB01",
            "name": "01_ontology_as_data_nodes_relationships.ipynb",
            "stage": "Ontology as Data",
            "status": "in_progress",
            "description": "Materializa a ontologia em CSVs de nós e relacionamentos.",
        },
        {
            "notebook_id": "NB02",
            "name": "02_graph_building_networkx_preview.ipynb",
            "stage": "Graph Building",
            "status": "planned",
            "description": "Constrói uma primeira visualização do grafo com Python.",
        },
        {
            "notebook_id": "NB03",
            "name": "03_neo4j_load_and_cypher_queries.ipynb",
            "stage": "Neo4j Load",
            "status": "planned",
            "description": "Prepara carga no Neo4j e primeiras consultas Cypher.",
        },
        {
            "notebook_id": "NB04",
            "name": "04_learning_progress_snapshots.ipynb",
            "stage": "Learning Analytics",
            "status": "planned",
            "description": "Analisa evolução temporal da jornada de aprendizagem.",
        },
    ]
)

labs = pd.DataFrame(
    [
        {
            "lab_id": "LAB001",
            "name": "Lakehouse Concept Map",
            "status": "planned",
            "description": "Mapa conceitual inicial da arquitetura Lakehouse.",
        },
        {
            "lab_id": "LAB002",
            "name": "Bronze Silver Gold Mini Pipeline",
            "status": "planned",
            "description": "Pipeline prático simulando arquitetura medalhão.",
        },
        {
            "lab_id": "LAB003",
            "name": "Delta Lake Transformation Lab",
            "status": "planned",
            "description": "Laboratório focado em tabelas Delta, MERGE e evolução de schema.",
        },
        {
            "lab_id": "LAB004",
            "name": "Databricks Workflow Lab",
            "status": "planned",
            "description": "Laboratório focado em jobs, tasks e orquestração.",
        },
    ]
)

skills = pd.DataFrame(
    [
        {
            "skill_id": "S001",
            "name": "Lakehouse Conceptual Modeling",
            "category": "Architecture",
            "priority": "high",
            "description": "Capacidade de explicar e modelar uma arquitetura Lakehouse.",
        },
        {
            "skill_id": "S002",
            "name": "Data Ingestion Design",
            "category": "Data Engineering",
            "priority": "high",
            "description": "Capacidade de desenhar processos de ingestão de dados.",
        },
        {
            "skill_id": "S003",
            "name": "Delta Lake Transformation",
            "category": "Data Engineering",
            "priority": "high",
            "description": "Capacidade de aplicar transformações usando Delta Lake.",
        },
        {
            "skill_id": "S004",
            "name": "Workflow Orchestration",
            "category": "Production",
            "priority": "high",
            "description": "Capacidade de estruturar pipelines produtivos com jobs e tasks.",
        },
        {
            "skill_id": "S005",
            "name": "Data Governance Awareness",
            "category": "Governance",
            "priority": "medium",
            "description": "Capacidade de aplicar princípios de governança e qualidade.",
        },
        {
            "skill_id": "S006",
            "name": "Learning Knowledge Graph Modeling",
            "category": "Knowledge Graph",
            "priority": "high",
            "description": "Capacidade de modelar uma jornada de aprendizagem como Knowledge Graph.",
        },
    ]
)


In [14]:
display(resources)
display(notebooks)
display(labs)
display(skills)

,resource_id,name,type,origin,status,description
0,R001,Databricks Academy,course_platform,official,planned,Plataforma oficial de cursos e trilhas de aprendizagem Databricks.
1,R002,Databricks Exam Guide,exam_guide,official,planned,Guia oficial de tópicos e estrutura da certificação.
2,R003,Databricks Documentation,documentation,official,planned,Documentação técnica da plataforma Databricks.
3,R004,Notebook 00 — Domain Understanding,internal_notebook,project,completed,Notebook conceitual inicial do projeto Databricks Learning KG.


,notebook_id,name,stage,status,description
0,NB00,00_domain_understanding_databricks_learning_kg.ipynb,Domain Understanding,completed,Define a ontologia inicial e o enquadramento CRISP-DM+.
1,NB01,01_ontology_as_data_nodes_relationships.ipynb,Ontology as Data,in_progress,Materializa a ontologia em CSVs de nós e relacionamentos.
2,NB02,02_graph_building_networkx_preview.ipynb,Graph Building,planned,Constrói uma primeira visualização do grafo com Python.
3,NB03,03_neo4j_load_and_cypher_queries.ipynb,Neo4j Load,planned,Prepara carga no Neo4j e primeiras consultas Cypher.
4,NB04,04_learning_progress_snapshots.ipynb,Learning Analytics,planned,Analisa evolução temporal da jornada de aprendizagem.


,lab_id,name,status,description
0,LAB001,Lakehouse Concept Map,planned,Mapa conceitual inicial da arquitetura Lakehouse.
1,LAB002,Bronze Silver Gold Mini Pipeline,planned,Pipeline prático simulando arquitetura medalhão.
2,LAB003,Delta Lake Transformation Lab,planned,"Laboratório focado em tabelas Delta, MERGE e evolução de schema."
3,LAB004,Databricks Workflow Lab,planned,"Laboratório focado em jobs, tasks e orquestração."


,skill_id,name,category,priority,description
0,S001,Lakehouse Conceptual Modeling,Architecture,high,Capacidade de explicar e modelar uma arquitetura Lakehouse.
1,S002,Data Ingestion Design,Data Engineering,high,Capacidade de desenhar processos de ingestão de dados.
2,S003,Delta Lake Transformation,Data Engineering,high,Capacidade de aplicar transformações usando Delta Lake.
3,S004,Workflow Orchestration,Production,high,Capacidade de estruturar pipelines produtivos com jobs e tasks.
4,S005,Data Governance Awareness,Governance,medium,Capacidade de aplicar princípios de governança e qualidade.
5,S006,Learning Knowledge Graph Modeling,Knowledge Graph,high,Capacidade de modelar uma jornada de aprendizagem como Knowledge Graph.


In [15]:
study_sessions = pd.DataFrame(
    [
        {
            "session_id": "SS001",
            "session_date": date.today().isoformat(),
            "duration_minutes": 60,
            "focus_area": "Ontology and Knowledge Graph setup",
            "energy_level": "high",
            "productivity_level": "high",
            "summary": "Início da materialização da ontologia do projeto Databricks Learning KG.",
            "next_action": "Gerar CSVs de nós e relacionamentos.",
        }
    ]
)

snapshots = pd.DataFrame(
    [
        {
            "snapshot_id": "SNAP_W00",
            "snapshot_date": date.today().isoformat(),
            "week_number": 0,
            "overall_progress": 0.05,
            "coverage_pct": 0.10,
            "avg_confidence": round(topics["confidence_score"].mean(), 4),
            "avg_mastery": round(topics["mastery_score"].mean(), 4),
            "main_gap": "Workflows, Unity Catalog e recursos específicos de Databricks ainda não praticados.",
            "main_achievement": "Ontologia inicial definida e convertida para dados estruturados.",
        }
    ]
)


In [16]:
display(study_sessions)
display(snapshots)

,session_id,session_date,duration_minutes,focus_area,energy_level,productivity_level,summary,next_action
0,SS001,2026-04-27,60,Ontology and Knowledge Graph setup,high,high,Início da materialização da ontologia do projeto Databricks Learning KG.,Gerar CSVs de nós e relacionamentos.


,snapshot_id,snapshot_date,week_number,overall_progress,coverage_pct,avg_confidence,avg_mastery,main_gap,main_achievement
0,SNAP_W00,2026-04-27,0,0.05,0.1,0.213,0.088,"Workflows, Unity Catalog e recursos específicos de Databricks ainda não praticados.",Ontologia inicial definida e convertida para dados estruturados.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 6. Relacionamentos estruturais</strong></font>

<font size="2" color='#66666'>

>- Nesta etapa criamos os relacionamentos principais:
    - certificação possui domínios;
    - domínios cobrem tópicos;
    - tópicos possuem subtópicos;
    - tópicos dependem de outros tópicos.

---

</font></div>

In [17]:
certification_domain_edges = make_edges(
    pairs=[
        {
            "source_id": "CERT_DEA_001",
            "target_id": row["domain_id"],
            "weight": row["weight_pct"] / 100,
            "description": f"A certificação cobre o domínio {row['name']}.",
        }
        for _, row in exam_domains.iterrows()
    ],
    source_label="Certification",
    relationship="HAS_DOMAIN",
    target_label="ExamDomain",
)

domain_topic_edges = make_edges(
    pairs=[
        {
            "source_id": row["domain_id"],
            "target_id": row["topic_id"],
            "weight": 1.0,
            "description": f"O domínio {row['domain_id']} cobre o tópico {row['name']}.",
        }
        for _, row in topics.iterrows()
    ],
    source_label="ExamDomain",
    relationship="COVERS",
    target_label="Topic",
)

topic_subtopic_edges = make_edges(
    pairs=[
        {
            "source_id": row["topic_id"],
            "target_id": row["subtopic_id"],
            "weight": 1.0,
            "description": f"O tópico {row['topic_id']} possui o subtópico {row['name']}.",
        }
        for _, row in subtopics.iterrows()
    ],
    source_label="Topic",
    relationship="HAS_SUBTOPIC",
    target_label="Subtopic",
)


In [18]:
display(certification_domain_edges.head())
display(domain_topic_edges.head())
display(topic_subtopic_edges.head())

,source_id,source_label,relationship,target_id,target_label,weight,description
0,CERT_DEA_001,Certification,HAS_DOMAIN,D01,ExamDomain,0.10,A certificação cobre o domínio Databricks Intelligence Platform.
1,CERT_DEA_001,Certification,HAS_DOMAIN,D02,ExamDomain,0.30,A certificação cobre o domínio Development and Ingestion.
2,CERT_DEA_001,Certification,HAS_DOMAIN,D03,ExamDomain,0.31,A certificação cobre o domínio Data Processing & Transformations.
3,CERT_DEA_001,Certification,HAS_DOMAIN,D04,ExamDomain,0.18,A certificação cobre o domínio Productionizing Data Pipelines.
4,CERT_DEA_001,Certification,HAS_DOMAIN,D05,ExamDomain,0.11,A certificação cobre o domínio Data Governance & Quality.


,source_id,source_label,relationship,target_id,target_label,weight,description
0,D01,ExamDomain,COVERS,T001,Topic,1.0,O domínio D01 cobre o tópico Lakehouse Architecture.
1,D01,ExamDomain,COVERS,T002,Topic,1.0,O domínio D01 cobre o tópico Databricks Workspace.
2,D01,ExamDomain,COVERS,T003,Topic,1.0,O domínio D01 cobre o tópico Compute.
3,D01,ExamDomain,COVERS,T004,Topic,1.0,O domínio D01 cobre o tópico SQL Warehouse.
4,D01,ExamDomain,COVERS,T005,Topic,1.0,O domínio D01 cobre o tópico Notebooks.


,source_id,source_label,relationship,target_id,target_label,weight,description
0,T001,Topic,HAS_SUBTOPIC,ST001,Subtopic,1.0,O tópico T001 possui o subtópico Data Lake vs Data Warehouse vs Lakehouse.
1,T001,Topic,HAS_SUBTOPIC,ST002,Subtopic,1.0,O tópico T001 possui o subtópico Lakehouse as unified architecture.
2,T016,Topic,HAS_SUBTOPIC,ST003,Subtopic,1.0,O tópico T016 possui o subtópico ACID transactions.
3,T016,Topic,HAS_SUBTOPIC,ST004,Subtopic,1.0,O tópico T016 possui o subtópico Delta transaction log.
4,T016,Topic,HAS_SUBTOPIC,ST005,Subtopic,1.0,O tópico T016 possui o subtópico Versioning.


In [19]:
prerequisite_pairs = [
    {"source_id": "T001", "target_id": "T020", "description": "Lakehouse Architecture ajuda a compreender Medallion Architecture."},
    {"source_id": "T016", "target_id": "T017", "description": "Delta Lake é pré-requisito para compreender Delta Table."},
    {"source_id": "T017", "target_id": "T021", "description": "Delta Table é pré-requisito para aplicar MERGE INTO."},
    {"source_id": "T007", "target_id": "T008", "description": "Data Ingestion antecede o uso de read_files."},
    {"source_id": "T007", "target_id": "T009", "description": "Data Ingestion antecede COPY INTO."},
    {"source_id": "T007", "target_id": "T010", "description": "Data Ingestion antecede Auto Loader."},
    {"source_id": "T012", "target_id": "T013", "description": "Inferência de schema ajuda a compreender definição explícita de schema."},
    {"source_id": "T014", "target_id": "T022", "description": "Spark SQL apoia o uso de window functions."},
    {"source_id": "T023", "target_id": "T024", "description": "Joins e agregações aparecem frequentemente em transformações analíticas."},
    {"source_id": "T029", "target_id": "T030", "description": "Databricks Workflows contextualiza Jobs."},
    {"source_id": "T030", "target_id": "T031", "description": "Jobs são compostos por Tasks."},
    {"source_id": "T031", "target_id": "T032", "description": "Tasks permitem dependências em pipelines produtivos."},
    {"source_id": "T038", "target_id": "T039", "description": "Unity Catalog organiza objetos como Catalog."},
    {"source_id": "T039", "target_id": "T040", "description": "Catalog contém Schemas."},
    {"source_id": "T040", "target_id": "T041", "description": "Schema contém Tables."},
]

prerequisite_edges = make_edges(
    pairs=prerequisite_pairs,
    source_label="Topic",
    relationship="PREREQUISITE_FOR",
    target_label="Topic",
)

prerequisite_edges.head()

,source_id,source_label,relationship,target_id,target_label,weight,description
0,T001,Topic,PREREQUISITE_FOR,T020,Topic,1.0,Lakehouse Architecture ajuda a compreender Medallion Architecture.
1,T016,Topic,PREREQUISITE_FOR,T017,Topic,1.0,Delta Lake é pré-requisito para compreender Delta Table.
2,T017,Topic,PREREQUISITE_FOR,T021,Topic,1.0,Delta Table é pré-requisito para aplicar MERGE INTO.
3,T007,Topic,PREREQUISITE_FOR,T008,Topic,1.0,Data Ingestion antecede o uso de read_files.
4,T007,Topic,PREREQUISITE_FOR,T009,Topic,1.0,Data Ingestion antecede COPY INTO.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 7. Relacionamentos de aprendizagem e portfólio</strong></font>

<font size="2" color='#66666'>

>- Agora conectamos recursos, notebooks, labs e skills aos tópicos.

>- Essas conexões serão fundamentais para responder perguntas como:
    - *Qual notebook comprova determinado conhecimento?*
    - *Qual recurso ensina determinado tópico?*
    - *Qual skill está sustentada por quais tópicos?*
    - *Onde estão minhas evidências de portfólio?*

---

</font></div>

In [20]:
notebook_topic_pairs = [
    {"source_id": "NB00", "target_id": "T001", "description": "Notebook 00 contextualiza Lakehouse Architecture."},
    {"source_id": "NB00", "target_id": "T020", "description": "Notebook 00 contextualiza Medallion Architecture."},
    {"source_id": "NB00", "target_id": "T038", "description": "Notebook 00 introduz governança via Unity Catalog."},
    {"source_id": "NB01", "target_id": "T001", "description": "Notebook 01 transforma a ontologia em dados."},
    {"source_id": "NB01", "target_id": "T020", "description": "Notebook 01 prepara a arquitetura medalhão como tópico rastreável."},
    {"source_id": "NB01", "target_id": "T045", "description": "Notebook 01 introduz rastreabilidade para qualidade de dados."},
    {"source_id": "NB02", "target_id": "T001", "description": "Notebook 02 deverá visualizar o grafo conceitual."},
    {"source_id": "NB03", "target_id": "T038", "description": "Notebook 03 deverá preparar carga e consultas em grafo."},
]

notebook_topic_edges = make_edges(
    pairs=notebook_topic_pairs,
    source_label="Notebook",
    relationship="PRACTICES",
    target_label="Topic",
)

lab_topic_pairs = [
    {"source_id": "LAB001", "target_id": "T001", "description": "Lab de mapa conceitual de Lakehouse."},
    {"source_id": "LAB002", "target_id": "T020", "description": "Lab de arquitetura Bronze, Silver e Gold."},
    {"source_id": "LAB003", "target_id": "T016", "description": "Lab de Delta Lake."},
    {"source_id": "LAB003", "target_id": "T021", "description": "Lab de MERGE INTO."},
    {"source_id": "LAB004", "target_id": "T029", "description": "Lab de Databricks Workflows."},
    {"source_id": "LAB004", "target_id": "T030", "description": "Lab de Jobs."},
    {"source_id": "LAB004", "target_id": "T031", "description": "Lab de Tasks."},
]

lab_topic_edges = make_edges(
    pairs=lab_topic_pairs,
    source_label="Lab",
    relationship="IMPLEMENTS",
    target_label="Topic",
)


In [21]:
display(notebook_topic_edges)
display(lab_topic_edges)

,source_id,source_label,relationship,target_id,target_label,weight,description
0,NB00,Notebook,PRACTICES,T001,Topic,1.0,Notebook 00 contextualiza Lakehouse Architecture.
1,NB00,Notebook,PRACTICES,T020,Topic,1.0,Notebook 00 contextualiza Medallion Architecture.
2,NB00,Notebook,PRACTICES,T038,Topic,1.0,Notebook 00 introduz governança via Unity Catalog.
3,NB01,Notebook,PRACTICES,T001,Topic,1.0,Notebook 01 transforma a ontologia em dados.
4,NB01,Notebook,PRACTICES,T020,Topic,1.0,Notebook 01 prepara a arquitetura medalhão como tópico rastreável.
5,NB01,Notebook,PRACTICES,T045,Topic,1.0,Notebook 01 introduz rastreabilidade para qualidade de dados.
6,NB02,Notebook,PRACTICES,T001,Topic,1.0,Notebook 02 deverá visualizar o grafo conceitual.
7,NB03,Notebook,PRACTICES,T038,Topic,1.0,Notebook 03 deverá preparar carga e consultas em grafo.


,source_id,source_label,relationship,target_id,target_label,weight,description
0,LAB001,Lab,IMPLEMENTS,T001,Topic,1.0,Lab de mapa conceitual de Lakehouse.
1,LAB002,Lab,IMPLEMENTS,T020,Topic,1.0,"Lab de arquitetura Bronze, Silver e Gold."
2,LAB003,Lab,IMPLEMENTS,T016,Topic,1.0,Lab de Delta Lake.
3,LAB003,Lab,IMPLEMENTS,T021,Topic,1.0,Lab de MERGE INTO.
4,LAB004,Lab,IMPLEMENTS,T029,Topic,1.0,Lab de Databricks Workflows.
5,LAB004,Lab,IMPLEMENTS,T030,Topic,1.0,Lab de Jobs.
6,LAB004,Lab,IMPLEMENTS,T031,Topic,1.0,Lab de Tasks.


In [22]:
resource_topic_pairs = [
    {"source_id": "R001", "target_id": "T001", "description": "Databricks Academy apoia fundamentos de Lakehouse."},
    {"source_id": "R001", "target_id": "T007", "description": "Databricks Academy apoia estudo de ingestão."},
    {"source_id": "R001", "target_id": "T016", "description": "Databricks Academy apoia estudo de Delta Lake."},
    {"source_id": "R001", "target_id": "T029", "description": "Databricks Academy apoia estudo de Workflows."},
    {"source_id": "R002", "target_id": "T014", "description": "Exam Guide orienta tópicos de Spark SQL."},
    {"source_id": "R002", "target_id": "T016", "description": "Exam Guide orienta tópicos de Delta Lake."},
    {"source_id": "R002", "target_id": "T038", "description": "Exam Guide orienta tópicos de governança."},
    {"source_id": "R003", "target_id": "T021", "description": "Documentação apoia estudo técnico de MERGE INTO."},
    {"source_id": "R003", "target_id": "T038", "description": "Documentação apoia estudo de Unity Catalog."},
    {"source_id": "R004", "target_id": "T001", "description": "Notebook 00 apoia entendimento conceitual da arquitetura."},
]

resource_topic_edges = make_edges(
    pairs=resource_topic_pairs,
    source_label="Resource",
    relationship="TEACHES",
    target_label="Topic",
)

skill_topic_pairs = [
    {"source_id": "S001", "target_id": "T001", "description": "Lakehouse Architecture sustenta modelagem conceitual."},
    {"source_id": "S001", "target_id": "T020", "description": "Medallion Architecture sustenta modelagem conceitual."},
    {"source_id": "S002", "target_id": "T007", "description": "Data Ingestion sustenta desenho de ingestão."},
    {"source_id": "S002", "target_id": "T008", "description": "read_files sustenta ingestão em Databricks."},
    {"source_id": "S003", "target_id": "T016", "description": "Delta Lake sustenta transformações confiáveis."},
    {"source_id": "S003", "target_id": "T021", "description": "MERGE INTO sustenta cargas incrementais."},
    {"source_id": "S004", "target_id": "T029", "description": "Workflows sustentam orquestração."},
    {"source_id": "S004", "target_id": "T030", "description": "Jobs sustentam produção."},
    {"source_id": "S005", "target_id": "T038", "description": "Unity Catalog sustenta governança."},
    {"source_id": "S005", "target_id": "T045", "description": "Data Quality sustenta governança."},
    {"source_id": "S006", "target_id": "T001", "description": "Modelagem semântica apoia aprendizagem estruturada."},
]

skill_topic_edges = make_edges(
    pairs=skill_topic_pairs,
    source_label="Skill",
    relationship="SUPPORTED_BY",
    target_label="Topic",
)


In [23]:
display(resource_topic_edges.head())
display(skill_topic_edges.head())

,source_id,source_label,relationship,target_id,target_label,weight,description
0,R001,Resource,TEACHES,T001,Topic,1.0,Databricks Academy apoia fundamentos de Lakehouse.
1,R001,Resource,TEACHES,T007,Topic,1.0,Databricks Academy apoia estudo de ingestão.
2,R001,Resource,TEACHES,T016,Topic,1.0,Databricks Academy apoia estudo de Delta Lake.
3,R001,Resource,TEACHES,T029,Topic,1.0,Databricks Academy apoia estudo de Workflows.
4,R002,Resource,TEACHES,T014,Topic,1.0,Exam Guide orienta tópicos de Spark SQL.


,source_id,source_label,relationship,target_id,target_label,weight,description
0,S001,Skill,SUPPORTED_BY,T001,Topic,1.0,Lakehouse Architecture sustenta modelagem conceitual.
1,S001,Skill,SUPPORTED_BY,T020,Topic,1.0,Medallion Architecture sustenta modelagem conceitual.
2,S002,Skill,SUPPORTED_BY,T007,Topic,1.0,Data Ingestion sustenta desenho de ingestão.
3,S002,Skill,SUPPORTED_BY,T008,Topic,1.0,read_files sustenta ingestão em Databricks.
4,S003,Skill,SUPPORTED_BY,T016,Topic,1.0,Delta Lake sustenta transformações confiáveis.


In [24]:
study_session_topic_pairs = [
    {"source_id": "SS001", "target_id": "T001", "description": "Sessão inicial estudou Lakehouse Architecture."},
    {"source_id": "SS001", "target_id": "T020", "description": "Sessão inicial estudou Medallion Architecture."},
    {"source_id": "SS001", "target_id": "T038", "description": "Sessão inicial introduziu Unity Catalog como área futura."},
    {"source_id": "SS001", "target_id": "T045", "description": "Sessão inicial posicionou qualidade como dimensão do KG."},
]

study_session_topic_edges = make_edges(
    pairs=study_session_topic_pairs,
    source_label="StudySession",
    relationship="STUDIED",
    target_label="Topic",
)

study_session_topic_edges

,source_id,source_label,relationship,target_id,target_label,weight,description
0,SS001,StudySession,STUDIED,T001,Topic,1.0,Sessão inicial estudou Lakehouse Architecture.
1,SS001,StudySession,STUDIED,T020,Topic,1.0,Sessão inicial estudou Medallion Architecture.
2,SS001,StudySession,STUDIED,T038,Topic,1.0,Sessão inicial introduziu Unity Catalog como área futura.
3,SS001,StudySession,STUDIED,T045,Topic,1.0,Sessão inicial posicionou qualidade como dimensão do KG.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 8. Snapshot inicial de progresso</strong></font>

<font size="2" color='#66666'>

>- O snapshot representa uma fotografia temporal da jornada.

>- Nesta primeira versão, usamos o snapshot da semana 0 para capturar o estado inicial de todos os tópicos.

---

</font></div>

In [25]:
topic_progress_snapshot = topics[
    [
        "topic_id",
        "name",
        "domain_id",
        "confidence_score",
        "mastery_score",
        "status",
        "priority",
    ]
].copy()

topic_progress_snapshot.insert(0, "snapshot_id", "SNAP_W00")
topic_progress_snapshot.insert(1, "snapshot_date", date.today().isoformat())
topic_progress_snapshot.insert(2, "week_number", 0)

topic_progress_snapshot.head()

,snapshot_id,snapshot_date,week_number,topic_id,name,domain_id,confidence_score,mastery_score,status,priority
0,SNAP_W00,2026-04-27,0,T001,Lakehouse Architecture,D01,0.25,0.10,initial_mapping,high
1,SNAP_W00,2026-04-27,0,T002,Databricks Workspace,D01,0.10,0.00,not_started,medium
2,SNAP_W00,2026-04-27,0,T003,Compute,D01,0.10,0.00,not_started,medium
3,SNAP_W00,2026-04-27,0,T004,SQL Warehouse,D01,0.10,0.00,not_started,medium
4,SNAP_W00,2026-04-27,0,T005,Notebooks,D01,0.50,0.25,initial_mapping,high


In [26]:
snapshot_topic_edges = make_edges(
    pairs=[
        {
            "source_id": "SNAP_W00",
            "target_id": row["topic_id"],
            "weight": row["mastery_score"],
            "description": f"Snapshot inicial captura o status do tópico {row['name']}.",
        }
        for _, row in topics.iterrows()
    ],
    source_label="Snapshot",
    relationship="CAPTURES_STATUS_OF",
    target_label="Topic",
)

snapshot_topic_edges.head()

,source_id,source_label,relationship,target_id,target_label,weight,description
0,SNAP_W00,Snapshot,CAPTURES_STATUS_OF,T001,Topic,0.10,Snapshot inicial captura o status do tópico Lakehouse Architecture.
1,SNAP_W00,Snapshot,CAPTURES_STATUS_OF,T002,Topic,0.00,Snapshot inicial captura o status do tópico Databricks Workspace.
2,SNAP_W00,Snapshot,CAPTURES_STATUS_OF,T003,Topic,0.00,Snapshot inicial captura o status do tópico Compute.
3,SNAP_W00,Snapshot,CAPTURES_STATUS_OF,T004,Topic,0.00,Snapshot inicial captura o status do tópico SQL Warehouse.
4,SNAP_W00,Snapshot,CAPTURES_STATUS_OF,T005,Topic,0.25,Snapshot inicial captura o status do tópico Notebooks.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 9. Consolidação dos nós</strong></font>

<font size="2" color='#66666'>

>- Agora vamos consolidar todas as tabelas de entidades em uma tabela única `nodes_all`.

>- Essa tabela será útil para:
    - auditoria;
    - visualização;
    - carga em ferramentas de grafos;
    - validação de relacionamentos órfãos.

---

</font></div>

In [27]:
nodes_all = pd.concat(
    [
        build_node_frame(
            certifications,
            id_col="certification_id",
            label="Certification",
            name_col="name",
            category_col="level",
            description_col="description",
            status_col="status",
        ),
        build_node_frame(
            exam_domains,
            id_col="domain_id",
            label="ExamDomain",
            name_col="name",
            category_col="priority",
            description_col="description",
            priority_col="priority",
        ),
        build_node_frame(
            topics,
            id_col="topic_id",
            label="Topic",
            name_col="name",
            category_col="category",
            status_col="status",
            priority_col="priority",
        ),
        build_node_frame(
            subtopics,
            id_col="subtopic_id",
            label="Subtopic",
            name_col="name",
            description_col="description",
        ),
        build_node_frame(
            resources,
            id_col="resource_id",
            label="Resource",
            name_col="name",
            category_col="type",
            description_col="description",
            status_col="status",
        ),
        build_node_frame(
            notebooks,
            id_col="notebook_id",
            label="Notebook",
            name_col="name",
            category_col="stage",
            description_col="description",
            status_col="status",
        ),
        build_node_frame(
            labs,
            id_col="lab_id",
            label="Lab",
            name_col="name",
            description_col="description",
            status_col="status",
        ),
        build_node_frame(
            skills,
            id_col="skill_id",
            label="Skill",
            name_col="name",
            category_col="category",
            description_col="description",
            priority_col="priority",
        ),
        build_node_frame(
            study_sessions,
            id_col="session_id",
            label="StudySession",
            name_col="focus_area",
            category_col="productivity_level",
            description_col="summary",
        ),
        build_node_frame(
            snapshots,
            id_col="snapshot_id",
            label="Snapshot",
            name_col="snapshot_date",
            category_col="week_number",
            description_col="main_achievement",
        ),
    ],
    ignore_index=True,
)


In [28]:
nodes_all.head(10)

,node_id,node_label,name,category,description,status,priority
0,CERT_DEA_001,Certification,Databricks Certified Data Engineer Associate,Associate,Certificação alvo para consolidar fundamentos de Engenharia de Dados na plataforma Databricks.,planned,
1,D01,ExamDomain,Databricks Intelligence Platform,medium,"Fundamentos da plataforma, workspace, compute e arquitetura Lakehouse.",,medium
2,D02,ExamDomain,Development and Ingestion,high,"Desenvolvimento, ingestão de dados e uso de SQL/PySpark.",,high
3,D03,ExamDomain,Data Processing & Transformations,high,"Transformações, Delta Lake, tabelas, joins, agregações e arquitetura medalhão.",,high
4,D04,ExamDomain,Productionizing Data Pipelines,high,"Jobs, Workflows, tarefas, agendamento, monitoramento e operação.",,high
5,D05,ExamDomain,Data Governance & Quality,medium,"Governança, Unity Catalog, qualidade, permissões e linhagem.",,medium
6,T001,Topic,Lakehouse Architecture,Platform,,initial_mapping,high
7,T002,Topic,Databricks Workspace,Platform,,not_started,medium
8,T003,Topic,Compute,Platform,,not_started,medium
9,T004,Topic,SQL Warehouse,Platform,,not_started,medium


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 10. Consolidação dos relacionamentos</strong></font>

<font size="2" color='#66666'>

- Agora vamos consolidar todas as arestas em uma tabela única `edges_all`.

- Essa tabela representa a espinha dorsal do grafo.

---

</font></div>

In [29]:
edges_all = pd.concat(
    [
        certification_domain_edges,
        domain_topic_edges,
        topic_subtopic_edges,
        prerequisite_edges,
        notebook_topic_edges,
        lab_topic_edges,
        resource_topic_edges,
        skill_topic_edges,
        study_session_topic_edges,
        snapshot_topic_edges,
    ],
    ignore_index=True,
)

edges_all.insert(0, "edge_id", [f"E{i:04d}" for i in range(1, len(edges_all) + 1)])


In [30]:
edges_all.head(10)

,edge_id,source_id,source_label,relationship,target_id,target_label,weight,description
0,E0001,CERT_DEA_001,Certification,HAS_DOMAIN,D01,ExamDomain,0.10,A certificação cobre o domínio Databricks Intelligence Platform.
1,E0002,CERT_DEA_001,Certification,HAS_DOMAIN,D02,ExamDomain,0.30,A certificação cobre o domínio Development and Ingestion.
2,E0003,CERT_DEA_001,Certification,HAS_DOMAIN,D03,ExamDomain,0.31,A certificação cobre o domínio Data Processing & Transformations.
3,E0004,CERT_DEA_001,Certification,HAS_DOMAIN,D04,ExamDomain,0.18,A certificação cobre o domínio Productionizing Data Pipelines.
4,E0005,CERT_DEA_001,Certification,HAS_DOMAIN,D05,ExamDomain,0.11,A certificação cobre o domínio Data Governance & Quality.
5,E0006,D01,ExamDomain,COVERS,T001,Topic,1.00,O domínio D01 cobre o tópico Lakehouse Architecture.
6,E0007,D01,ExamDomain,COVERS,T002,Topic,1.00,O domínio D01 cobre o tópico Databricks Workspace.
7,E0008,D01,ExamDomain,COVERS,T003,Topic,1.00,O domínio D01 cobre o tópico Compute.
8,E0009,D01,ExamDomain,COVERS,T004,Topic,1.00,O domínio D01 cobre o tópico SQL Warehouse.
9,E0010,D01,ExamDomain,COVERS,T005,Topic,1.00,O domínio D01 cobre o tópico Notebooks.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 11. Validação de qualidade</strong></font>

<font size="2" color='#66666'>

>- Antes de exportar os CSVs, vamos validar:
    - duplicidade de IDs de nós;
    - relacionamentos com origem inexistente;
    - relacionamentos com destino inexistente;
    - quantidade de nós por tipo;
    - quantidade de relacionamentos por tipo.

---

</font></div>

In [31]:
duplicated_nodes = nodes_all[nodes_all.duplicated(subset=["node_id"], keep=False)]

node_ids = set(nodes_all["node_id"])

orphan_sources = sorted(set(edges_all["source_id"]) - node_ids)
orphan_targets = sorted(set(edges_all["target_id"]) - node_ids)

qa_summary = pd.DataFrame(
    [
        {
            "check": "Duplicated node IDs",
            "result": len(duplicated_nodes),
            "status": "OK" if len(duplicated_nodes) == 0 else "REVIEW",
        },
        {
            "check": "Orphan source IDs",
            "result": len(orphan_sources),
            "status": "OK" if len(orphan_sources) == 0 else "REVIEW",
        },
        {
            "check": "Orphan target IDs",
            "result": len(orphan_targets),
            "status": "OK" if len(orphan_targets) == 0 else "REVIEW",
        },
        {
            "check": "Total nodes",
            "result": len(nodes_all),
            "status": "INFO",
        },
        {
            "check": "Total edges",
            "result": len(edges_all),
            "status": "INFO",
        },
    ]
)

qa_summary

,check,result,status
0,Duplicated node IDs,0,OK
1,Orphan source IDs,0,OK
2,Orphan target IDs,0,OK
3,Total nodes,93,INFO
4,Total edges,172,INFO


In [32]:
nodes_by_label = (
    nodes_all
    .groupby("node_label", as_index=False)
    .agg(total_nodes=("node_id", "count"))
    .sort_values("total_nodes", ascending=False)
)

edges_by_type = (
    edges_all
    .groupby("relationship", as_index=False)
    .agg(total_edges=("edge_id", "count"))
    .sort_values("total_edges", ascending=False)
)


In [33]:
display(nodes_by_label)
display(edges_by_type)

,node_label,total_nodes
9,Topic,46
8,Subtopic,20
5,Skill,6
1,ExamDomain,5
3,Notebook,5
4,Resource,4
2,Lab,4
0,Certification,1
7,StudySession,1
6,Snapshot,1


,relationship,total_edges
0,CAPTURES_STATUS_OF,46
1,COVERS,46
3,HAS_SUBTOPIC,20
6,PREREQUISITE_FOR,15
8,SUPPORTED_BY,11
9,TEACHES,10
5,PRACTICES,8
4,IMPLEMENTS,7
2,HAS_DOMAIN,5
7,STUDIED,4


In [34]:
assert duplicated_nodes.empty, "Existem IDs duplicados em nodes_all."
assert len(orphan_sources) == 0, f"Existem source_ids órfãos: {orphan_sources}"
assert len(orphan_targets) == 0, f"Existem target_ids órfãos: {orphan_targets}"

print("Validação concluída com sucesso.")

Validação concluída com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 12. Exportação dos CSVs</strong></font>

<font size="2" color='#66666'>

>- Nesta etapa, salvaremos os arquivos em:
    - `data/processed/`

>- Esses arquivos serão usados nos próximos notebooks para visualização, carga em grafo e análise de evolução.

---

</font></div>

In [35]:
tables_to_export = {
    "certifications.csv": certifications,
    "exam_domains.csv": exam_domains,
    "topics.csv": topics,
    "subtopics.csv": subtopics,
    "resources.csv": resources,
    "notebooks.csv": notebooks,
    "labs.csv": labs,
    "skills.csv": skills,
    "study_sessions.csv": study_sessions,
    "snapshots.csv": snapshots,
    "topic_progress_snapshot.csv": topic_progress_snapshot,
    "nodes_all.csv": nodes_all,
    "edges_all.csv": edges_all,
    "relationship_types_summary.csv": edges_by_type,
    "node_labels_summary.csv": nodes_by_label,
}

manifest_rows = []

for filename, df in tables_to_export.items():
    manifest_rows.append(save_table(df, filename))

export_manifest = pd.DataFrame(manifest_rows)

manifest_path = PROCESSED_DIR / "export_manifest.csv"
export_manifest.to_csv(manifest_path, index=False, encoding="utf-8")

export_manifest

,filename,path,rows,columns,export_date
0,certifications.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\certifications.csv,1,10,2026-04-27
1,exam_domains.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\exam_domains.csv,5,5,2026-04-27
2,topics.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\topics.csv,46,12,2026-04-27
3,subtopics.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\subtopics.csv,20,4,2026-04-27
4,resources.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\resources.csv,4,6,2026-04-27
5,notebooks.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\notebooks.csv,5,5,2026-04-27
6,labs.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\labs.csv,4,4,2026-04-27
7,skills.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\skills.csv,6,5,2026-04-27
8,study_sessions.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\study_sessions.csv,1,8,2026-04-27
9,snapshots.csv,D:\_DS-Projects\Data-Science\databricks-learning-kg\data\processed\snapshots.csv,1,9,2026-04-27


In [36]:
sorted([path.name for path in PROCESSED_DIR.glob("*.csv")])

['certifications.csv',
 'edges_all.csv',
 'exam_domains.csv',
 'export_manifest.csv',
 'labs.csv',
 'node_labels_summary.csv',
 'nodes_all.csv',
 'notebooks.csv',
 'relationship_types_summary.csv',
 'resources.csv',
 'skills.csv',
 'snapshots.csv',
 'study_sessions.csv',
 'subtopics.csv',
 'topic_progress_snapshot.csv',
 'topics.csv']

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 13. Documentação automática da ontologia</strong></font>

<font size="2" color='#66666'>

>- Além dos CSVs, vamos gerar uma documentação simples em Markdown com os principais números do grafo inicial.

---

</font></div>

In [40]:
ontology_doc = f"""# Ontology v0.1 — Databricks Learning Knowledge Graph

## Projeto

{PROJECT_NAME}

## Certificação alvo

{CERTIFICATION_NAME}

## Data de geração

{date.today().isoformat()}

## Visão geral

Este documento descreve a primeira versão materializada da ontologia do projeto.

## Quantidade de nós por tipo

{nodes_by_label.to_markdown(index=False)}

## Quantidade de relacionamentos por tipo

{edges_by_type.to_markdown(index=False)}

## Arquivos gerados

{export_manifest[["filename", "rows", "columns"]].to_markdown(index=False)}

## Observação

Esta é a versão inicial da ontologia como dados.  
Nos próximos notebooks, essa estrutura poderá ser visualizada como grafo, carregada no Neo4j e enriquecida com novos snapshots de progresso.
"""

ontology_path = DOCS_DIR / "ontology_v01.md"
ontology_path.write_text(ontology_doc, encoding="utf-8")

ontology_path

WindowsPath('D:/_DS-Projects/Data-Science/databricks-learning-kg/docs/ontology_v01.md')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 14. Conclusão executiva</strong></font>

<font size="2" color='#66666'>

>- Este notebook materializou a ontologia inicial do projeto Databricks Certification Learning Knowledge Graph.

>- Foram criados:
    - nós de certificação;
    - domínios do exame;
    - tópicos técnicos;
    - subtópicos;
    - recursos;
    - notebooks;
    - labs;
    - skills;
    - sessão inicial de estudo;
    - snapshot inicial;
    - relacionamentos estruturais;
    - relacionamentos de aprendizagem;
    - relacionamentos de portfólio;
    - CSV consolidado de nós;
    - CSV consolidado de relacionamentos;
    - manifesto de exportação;
    - documentação Markdown da ontologia.

>- **Próximo passo**
    - O próximo notebook será:
    - `Notebook 02 — Graph Building: visualização inicial do KG com Python`
    - Nele vamos carregar `nodes_all.csv` e `edges_all.csv`, construir o grafo com NetworkX e gerar uma primeira visualização navegável.

---

</font></div>

In [41]:
%reload_ext watermark 
%watermark -a "Roberto-SSoares-LfLngLrnng" -d -t -u --iversions -v -m -h

Author: Roberto-SSoares-LfLngLrnng

Last updated: 2026-04-27 16:41:58

Python implementation: CPython
Python version       : 3.12.12
IPython version      : 9.13.0

Compiler    : MSC v.1944 64 bit (AMD64)
OS          : Windows
Release     : 11
Machine     : AMD64
Processor   : Intel64 Family 6 Model 158 Stepping 9, GenuineIntel
CPU cores   : 4
Architecture: 64bit

Hostname: PC-ROBERTO

pandas  : 3.0.2
tabulate: 0.10.0



<div style="background-color:#f3f2ee">
    
<font size="6" color='#CC403E'><strong>Fim</strong></font>

<font size="2" color='#66666'></font></div>

In [46]:
#!uv pip install nbconvert -U -q
!jupyter nbconvert --to html --template-file my-template-html-v10.tpl 01_ontology_as_data_nodes_relationships.ipynb

[NbConvertApp] Converting notebook 01_ontology_as_data_nodes_relationships.ipynb to html
[NbConvertApp] Writing 108808 bytes to 01_ontology_as_data_nodes_relationships.html
